# 01 — Dataset Audit

## 1 — Environment Verification

This section verifies the Python environment used for the analysis and records the location of the current research project.

In [1]:
import sys
from pathlib import Path

print("Python:", sys.executable)
print("My project folder:", Path.cwd())

Python: C:\Users\rahin\OneDrive\Desktop\TU\Summer 2026\Brain fmri in R\Selim code\gnn_env\Scripts\python.exe
My project folder: C:\Users\rahin\OneDrive\Desktop\TU\Summer 2026\Brain fmri in R\neurofeedback-gnn


## 2 — Dataset Location

This section connects the current analysis project to the resting-state fMRI dataset without copying or modifying the original imaging files.

The dataset contains two directories, referred to as **Rest 1** and **Rest 2**. At this stage, these names are treated only as session labels; their experimental interpretation will be verified separately.

In [3]:
from pathlib import Path

PROJECT_DIR = Path.cwd()

DATA_DIR = Path(
    r"C:\Neurofeedback_Data_Fall26\Imaging\Processed Rest1&2 Scans"
)

REST1_DIR = DATA_DIR / "Processed rest scans"
REST2_DIR = DATA_DIR / "Processed rest2 scans"

print("Project folder:", PROJECT_DIR)
print("Data folder:", DATA_DIR)
print("Dataset exists:", DATA_DIR.exists())
print("Rest 1 exists:", REST1_DIR.exists())
print("Rest 2 exists:", REST2_DIR.exists())

Project folder: C:\Users\rahin\OneDrive\Desktop\TU\Summer 2026\Brain fmri in R\neurofeedback-gnn
Data folder: C:\Neurofeedback_Data_Fall26\Imaging\Processed Rest1&2 Scans
Dataset exists: True
Rest 1 exists: True
Rest 2 exists: True


## 3 — Participant Inventory

This section identifies the unique participant exam IDs represented in each resting-state session.

The goal is to determine:

- how many participants are available in Rest 1,
- how many are available in Rest 2,
- and whether the same participants appear in both sessions.

No treatment-group or clinical labels are assigned at this stage.

In [5]:
import re

def extract_exam_ids(folder):
    exam_ids = set()

    for path in folder.rglob("*"):
        match = re.search(r"E\d{4}", str(path))
        if match:
            exam_ids.add(match.group())

    return sorted(exam_ids)

rest1_subjects = extract_exam_ids(REST1_DIR)
rest2_subjects = extract_exam_ids(REST2_DIR)

print("Rest 1 subjects:", len(rest1_subjects))
print(rest1_subjects)

print("\nRest 2 subjects:", len(rest2_subjects))
print(rest2_subjects)

Rest 1 subjects: 23
['E3746', 'E3799', 'E3834', 'E3973', 'E4051', 'E4209', 'E4253', 'E4324', 'E4350', 'E4360', 'E4484', 'E4673', 'E4689', 'E4696', 'E4697', 'E4745', 'E5215', 'E5376', 'E5580', 'E5586', 'E5653', 'E5693', 'E5694']

Rest 2 subjects: 22
['E3746', 'E3799', 'E3973', 'E4051', 'E4209', 'E4253', 'E4324', 'E4350', 'E4360', 'E4484', 'E4673', 'E4689', 'E4696', 'E4697', 'E4745', 'E5215', 'E5376', 'E5580', 'E5586', 'E5653', 'E5693', 'E5694']


## 4 — Paired Session Availability

Participant IDs from Rest 1 and Rest 2 are compared to identify participants with repeated resting-state measurements.

Participants with data from both sessions may support paired or longitudinal analyses, while participants with only one available session are documented separately.

At this stage, the experimental meaning of Rest 1 and Rest 2 is not assumed.

In [7]:
rest1_set = set(rest1_subjects)
rest2_set = set(rest2_subjects)

both_sessions = sorted(rest1_set & rest2_set)
rest1_only = sorted(rest1_set - rest2_set)
rest2_only = sorted(rest2_set - rest1_set)

print("Participants with BOTH sessions:", len(both_sessions))
print(both_sessions)

print("\nRest 1 only:", len(rest1_only))
print(rest1_only)

print("\nRest 2 only:", len(rest2_only))
print(rest2_only)

Participants with BOTH sessions: 22
['E3746', 'E3799', 'E3973', 'E4051', 'E4209', 'E4253', 'E4324', 'E4350', 'E4360', 'E4484', 'E4673', 'E4689', 'E4696', 'E4697', 'E4745', 'E5215', 'E5376', 'E5580', 'E5586', 'E5653', 'E5693', 'E5694']

Rest 1 only: 1
['E3834']

Rest 2 only: 0
[]


### Interpretation

The dataset contains 23 unique participant identifiers in Rest 1 and 22 in Rest 2.

A total of **22 participants have data represented in both resting-state sessions**, while **E3834 is represented only in Rest 1**. No participant is represented exclusively in Rest 2.

These counts describe file/session availability only. They do not yet establish that all 22 paired participants have complete or analysis-quality fMRI data. Imaging-file completeness and quality-control exclusions are therefore evaluated next.

## 5 — AFNI Imaging File Completeness

The presence of a participant identifier within a session directory does not necessarily indicate that a complete fMRI dataset is available.

AFNI datasets are represented by paired `.HEAD` and `.BRIK` (or compressed `.BRIK.gz`) files. This section identifies the residual fMRI datasets and determines whether the required imaging components are available for each participant and session.

This provides a file-level quality-control step before ROI extraction or functional-connectivity analysis.

In [17]:
from pathlib import Path
import re
import pandas as pd

def audit_afni_session(folder, session_name):
    records = []

    head_files = [
        p for p in folder.rglob("*.HEAD")
        if "errts." in p.name.lower()
    ]

    for head in head_files:
        name = head.name.lower()

        # Keep only files belonging to the requested session
        if session_name == "Rest1" and "rest2" in name:
            continue

        if session_name == "Rest2" and "rest2" not in name:
            continue

        match = re.search(r"E\d{4}", str(head))
        if not match:
            continue

        subject_id = match.group()

        base = str(head)[:-5]
        brik = Path(base + ".BRIK")
        brik_gz = Path(base + ".BRIK.gz")

        records.append({
            "subject_id": subject_id,
            "session": session_name,
            "HEAD": True,
            "BRIK": brik.exists(),
            "BRIK_gz": brik_gz.exists(),
            "complete_pair": brik.exists() or brik_gz.exists(),
            "HEAD_file": head.name
        })

    return pd.DataFrame(records)

In [23]:
rest1_afni = audit_afni_session(REST1_DIR, "Rest1")
rest2_afni = audit_afni_session(REST2_DIR, "Rest2")

print("REST 1")
print("Residual HEAD datasets:", len(rest1_afni))
print("Complete HEAD/BRIK pairs:", rest1_afni["complete_pair"].sum())

print("\nREST 2")
print("Residual HEAD datasets:", len(rest2_afni))
print("Complete HEAD/BRIK pairs:", rest2_afni["complete_pair"].sum())

REST 1
Residual HEAD datasets: 23
Complete HEAD/BRIK pairs: 23

REST 2
Residual HEAD datasets: 23
Complete HEAD/BRIK pairs: 23


In [6]:
%pip install pandas

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ------------------------------------- -- 9.2/9.8 MB 51.8 MB/s eta 0:00:01
   ---------------------------------------  9.7/9.8 MB 32.4 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 22.6 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\rahin\OneDrive\Desktop\TU\Summer 2026\Brain fmri in R\Selim code\gnn_env\Scripts\python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
print("pandas:", pd.__version__)

pandas: 3.0.5


### Interpretation

The corrected local dataset contains complete AFNI HEAD/BRIK pairs for all
23 Rest 2 residual datasets.

For Rest 1, 24 residual HEAD datasets were identified, of which 21 currently
have matching BRIK files.

Because the number of Rest 1 HEAD datasets exceeds the number of unique
participant IDs, participant-level completeness and possible duplicate datasets
must be examined before defining the final paired analysis cohort.

## 6 — Missing AFNI Data by Participant

This section identifies which participants currently lack complete residual AFNI imaging pairs in each session.

The purpose is to distinguish missing or unavailable imaging files from later quality-control exclusions such as excessive motion.

In [25]:
def summarize_incomplete(df):
    if df.empty:
        return []

    incomplete = df.loc[~df["complete_pair"], "subject_id"]
    return sorted(incomplete.unique())

rest1_incomplete = summarize_incomplete(rest1_afni)
rest2_incomplete = summarize_incomplete(rest2_afni)

print("Rest 1 incomplete subjects:", len(rest1_incomplete))
print(rest1_incomplete)

print("\nRest 2 incomplete subjects:", len(rest2_incomplete))
print(rest2_incomplete)

Rest 1 incomplete subjects: 0
[]

Rest 2 incomplete subjects: 0
[]


### File-Availability Finding

All identified Rest 1 and Rest 2 residual AFNI datasets currently have
complete HEAD/BRIK pairs.

No participants are excluded at this stage based solely on AFNI file
availability. Participant pairing and subsequent quality-control criteria
will be evaluated separately before defining the final analysis cohort.

## 7 — Final Paired Imaging Cohort

In [27]:
rest1_complete_ids = set(
    rest1_afni.loc[rest1_afni["complete_pair"], "subject_id"]
)

rest2_complete_ids = set(
    rest2_afni.loc[rest2_afni["complete_pair"], "subject_id"]
)

paired_complete_ids = sorted(
    rest1_complete_ids.intersection(rest2_complete_ids)
)

rest1_only_complete = sorted(
    rest1_complete_ids - rest2_complete_ids
)

rest2_only_complete = sorted(
    rest2_complete_ids - rest1_complete_ids
)

print("Complete Rest 1 participants:", len(rest1_complete_ids))
print("Complete Rest 2 participants:", len(rest2_complete_ids))
print("Complete paired participants:", len(paired_complete_ids))

print("\nRest 1 only:")
print(rest1_only_complete)

print("\nRest 2 only:")
print(rest2_only_complete)

Complete Rest 1 participants: 23
Complete Rest 2 participants: 22
Complete paired participants: 22

Rest 1 only:
['E3834']

Rest 2 only:
[]


### Paired-Imaging Finding

Twenty-two participants currently have complete residual AFNI imaging
datasets for both Rest 1 and Rest 2.

Participant E3834 has a complete Rest 1 dataset but no corresponding
Rest 2 dataset in the current paired imaging collection and therefore
cannot contribute to paired pre-post analyses.

This 22-participant set represents the file-complete paired imaging cohort,
not the final analysis cohort. Subsequent metadata integration and
quality-control assessment will determine the final eligible sample.

8 — Participant Metadata Integration

In [29]:
participants_file = DATA_DIR / "participants.tsv"

participants = pd.read_csv(participants_file, sep="\t")

print("Metadata file exists:", participants_file.exists())
print("Metadata rows:", len(participants))
print("Metadata columns:", len(participants.columns))

print("\nColumns:")
print(participants.columns.tolist())

Metadata file exists: True
Metadata rows: 355
Metadata columns: 27

Columns:
['Sub', 'Group', 'Age', 'Sex', 'Dx', 'Study', 'Datetime', 'Exam', 'Ses', 'anat', 'rtrest', 'rtbaseline', 'rtpractice', 'rttraining1', 'rttraining2', 'rttraining3', 'rttransfer', 'rtrest2', 'MADRS', 'MADRS_post', 'HAMD', 'HAMD_post', 'PHQ', 'PHQ_post', 'post_days', 'PCLM', 'PCLM_post']


In [31]:
# Match metadata to the 22 participants with paired imaging

paired_metadata = participants[
    participants["Exam"].isin(paired_complete_ids)
].copy()

metadata_exam_ids = set(paired_metadata["Exam"].dropna().astype(str))
paired_id_set = set(paired_complete_ids)

missing_from_metadata = sorted(paired_id_set - metadata_exam_ids)

print("Paired imaging participants:", len(paired_complete_ids))
print("Matched metadata rows:", len(paired_metadata))
print("Unique matched Exam IDs:", paired_metadata["Exam"].nunique())

print("\nImaging IDs missing from metadata:")
print(missing_from_metadata)

print("\nGroup counts:")
print(
    paired_metadata[["Exam", "Group"]]
    .drop_duplicates()["Group"]
    .value_counts(dropna=False)
)

print("\nDiagnosis counts:")
print(
    paired_metadata[["Exam", "Dx"]]
    .drop_duplicates()["Dx"]
    .value_counts(dropna=False)
)

Paired imaging participants: 22
Matched metadata rows: 22
Unique matched Exam IDs: 22

Imaging IDs missing from metadata:
[]

Group counts:
Group
active    11
sham      11
Name: count, dtype: int64

Diagnosis counts:
Dx
MDD    22
Name: count, dtype: int64


### Metadata-Matching Finding

All 22 participants in the file-complete paired imaging cohort were
successfully matched to the study metadata.

The paired cohort contains 11 participants assigned to the active
neurofeedback condition and 11 assigned to the sham condition. All 22
matched participants have an MDD diagnosis in the metadata.

These counts describe the metadata-linked paired imaging cohort prior to
quality-control exclusions.

## 9 — Imaging Quality Control and Final Analysis Cohort

This section evaluates motion-related quality-control information for the
paired Rest 1 and Rest 2 scans.

The purpose is to distinguish participants with available imaging data from
participants whose scans contain sufficient usable data for the final
pre-post analysis.

### 9.1 Motion-QC File Inventory

This step identifies the AFNI motion-censor files available for each Rest 1
and Rest 2 scan. These files will be used to quantify the number of usable
fMRI volumes after motion censoring.

In [33]:
def find_censor_files(folder, session_name):
    records = []

    for p in folder.rglob("*censor.1D"):
        name = p.name.lower()

        # Keep the correct session only
        if session_name == "Rest1" and "rest2" in name:
            continue

        if session_name == "Rest2" and "rest2" not in name:
            continue

        match = re.search(r"E\d{4}", str(p))
        if match:
            records.append({
                "subject_id": match.group(),
                "session": session_name,
                "censor_file": str(p)
            })

    return pd.DataFrame(records)


rest1_censor = find_censor_files(REST1_DIR, "Rest1")
rest2_censor = find_censor_files(REST2_DIR, "Rest2")

print("Rest 1 censor files:", len(rest1_censor))
print("Rest 1 unique participants:", rest1_censor["subject_id"].nunique())

print("\nRest 2 censor files:", len(rest2_censor))
print("Rest 2 unique participants:", rest2_censor["subject_id"].nunique())

Rest 1 censor files: 23
Rest 1 unique participants: 23

Rest 2 censor files: 23
Rest 2 unique participants: 22


### 9.2 Duplicate QC-File Investigation

The motion-QC inventory identified one additional Rest 2 censor file.
This step determines which participant has multiple candidate Rest 2
processing outputs so that the correct longitudinal scan can be selected.

In [35]:
rest2_censor_counts = (
    rest2_censor["subject_id"]
    .value_counts()
    .sort_values(ascending=False)
)

print("Rest 2 censor-file counts per participant:")
print(rest2_censor_counts)

print("\nParticipants with more than one Rest 2 censor file:")
print(rest2_censor_counts[rest2_censor_counts > 1])

Rest 2 censor-file counts per participant:
subject_id
E3746    2
E5694    1
E4484    1
E4689    1
E4673    1
E4697    1
E4696    1
E3799    1
E5580    1
E4324    1
E5376    1
E5693    1
E4360    1
E4745    1
E3973    1
E4209    1
E5653    1
E5586    1
E5215    1
E4051    1
E4253    1
E4350    1
Name: count, dtype: int64

Participants with more than one Rest 2 censor file:
subject_id
E3746    2
Name: count, dtype: int64


In [37]:
e3746_rest2 = rest2_censor[
    rest2_censor["subject_id"] == "E3746"
]

for path in e3746_rest2["censor_file"]:
    print(path)

C:\Neurofeedback_Data_Fall26\Imaging\Processed Rest1&2 Scans\Processed rest2 scans\DP_E3746_rest2.results\motion_DP_E3746_rest2_censor.1D
C:\Neurofeedback_Data_Fall26\Imaging\Processed Rest1&2 Scans\Processed rest2 scans\TP_E3746_rest2.results\motion_TP_E3746_rest2_censor.1D


In [39]:
e3746_meta = participants[
    participants["Exam"].astype(str) == "E3746"
]

print(
    e3746_meta[
        ["Sub", "Exam", "Ses", "Group", "Dx", "rtrest", "rtrest2"]
    ].to_string(index=False)
)

  Sub  Exam        Ses  Group  Dx rtrest rtrest2
AA341 E3746 2rtfeedMDD active MDD scan_6 scan_12


In [41]:
for folder_name in [
    "DP_E3746_rest2.results",
    "TP_E3746_rest2.results"
]:
    folder = REST2_DIR / folder_name

    print(f"\n===== {folder_name} =====")

    if not folder.exists():
        print("Folder not found")
        continue

    for p in sorted(folder.iterdir()):
        if (
            "errts" in p.name.lower()
            or "censor" in p.name.lower()
            or "enorm" in p.name.lower()
        ):
            print(p.name)


===== DP_E3746_rest2.results =====
errts.DP_E3746_rest2.fanaticor+tlrc.BRIK
errts.DP_E3746_rest2.fanaticor+tlrc.HEAD
motion_DP_E3746_rest2_censor.1D
motion_DP_E3746_rest2_CENSORTR.txt
motion_DP_E3746_rest2_enorm.1D

===== TP_E3746_rest2.results =====
errts.TP_E3746_rest2.fanaticor+tlrc.BRIK
errts.TP_E3746_rest2.fanaticor+tlrc.HEAD
motion_TP_E3746_rest2_censor.1D
motion_TP_E3746_rest2_CENSORTR.txt
motion_TP_E3746_rest2_enorm.1D


In [43]:
for p in sorted(REST1_DIR.rglob("*E3746*")):
    if p.is_dir():
        print(p)

C:\Neurofeedback_Data_Fall26\Imaging\Processed Rest1&2 Scans\Processed rest scans\E3746


In [45]:
e3746_rest1_folder = REST1_DIR / "E3746"

print("===== E3746 REST 1 =====")

for p in e3746_rest1_folder.rglob("*"):
    if (
        "errts" in p.name.lower()
        or "censor" in p.name.lower()
        or "enorm" in p.name.lower()
    ):
        print(p.relative_to(e3746_rest1_folder))

===== E3746 REST 1 =====
errts.DP_E3746_rest.fanaticor+tlrc.BRIK
errts.DP_E3746_rest.fanaticor+tlrc.HEAD
motion_DP_E3746_rest_censor.1D
motion_DP_E3746_rest_CENSORTR.txt
motion_DP_E3746_rest_enorm.1D


In [47]:
# Show residual dataset names for every paired participant

print("===== REST 1 =====")
for sid in paired_complete_ids:
    files = [
        p.name for p in REST1_DIR.rglob("*.HEAD")
        if sid in str(p)
        and "errts." in p.name.lower()
        and "rest2" not in p.name.lower()
    ]
    print(sid, "->", files)

print("\n===== REST 2 =====")
for sid in paired_complete_ids:
    files = [
        p.name for p in REST2_DIR.rglob("*.HEAD")
        if sid in str(p)
        and "errts." in p.name.lower()
        and "rest2" in p.name.lower()
    ]
    print(sid, "->", files)

===== REST 1 =====
E3746 -> ['errts.DP_E3746_rest.fanaticor+tlrc.HEAD']
E3799 -> ['errts.HL_E3799_rest.fanaticor+tlrc.HEAD']
E3973 -> ['errts.LD_E3973_rest.fanaticor+tlrc.HEAD']
E4051 -> ['errts.SA_E4051_rest.fanaticor+tlrc.HEAD']
E4209 -> ['errts.LS_E4209_rest.fanaticor+tlrc.HEAD']
E4253 -> ['errts.SA_E4253_rest.fanaticor+tlrc.HEAD']
E4324 -> ['errts.JV_E4324_rest.fanaticor+tlrc.HEAD']
E4350 -> ['errts.SF_E4350_rest.fanaticor+tlrc.HEAD']
E4360 -> ['errts.KS_E4360_rest.fanaticor+tlrc.HEAD']
E4484 -> ['errts.CB_E4484_rest.fanaticor+tlrc.HEAD']
E4673 -> ['errts.CL_E4673_rest.fanaticor+tlrc.HEAD']
E4689 -> ['errts.CC_E4689_rest.fanaticor+tlrc.HEAD']
E4696 -> ['errts.DH_E4696_rest.fanaticor+tlrc.HEAD']
E4697 -> ['errts.CM_E4697_rest.fanaticor+tlrc.HEAD']
E4745 -> ['errts.LA_E4745_rest.fanaticor+tlrc.HEAD']
E5215 -> ['errts.RS_E5215_rest.fanaticor+tlrc.HEAD']
E5376 -> ['errts.KE_E5376_rest.fanaticor+tlrc.HEAD']
E5580 -> ['errts.IR_E5580_rest.fanaticor+tlrc.HEAD']
E5586 -> ['errts.PS_E5586_r

### 9.3 Paired Scan Manifest

In [49]:
import re
from pathlib import Path

records = []

for sid in paired_complete_ids:
    # Rest 1 candidates
    rest1_heads = [
        p for p in REST1_DIR.rglob("*.HEAD")
        if sid in str(p)
        and "errts." in p.name.lower()
        and "rest2" not in p.name.lower()
    ]

    if len(rest1_heads) != 1:
        print(f"WARNING Rest1 {sid}: {len(rest1_heads)} candidates")
        continue

    rest1_head = rest1_heads[0]

    # Extract prefix before the exam ID, e.g. DP, HL, KH
    m = re.search(r"errts\.([A-Za-z]+)_" + sid, rest1_head.name)
    prefix = m.group(1) if m else None

    # Rest 2 candidates
    rest2_heads = [
        p for p in REST2_DIR.rglob("*.HEAD")
        if sid in str(p)
        and "errts." in p.name.lower()
        and "rest2" in p.name.lower()
    ]

    # Keep the Rest2 file with the same prefix as Rest1
    matching_rest2 = [
        p for p in rest2_heads
        if prefix and f"errts.{prefix}_{sid}" in p.name
    ]

    if len(matching_rest2) != 1:
        print(f"WARNING Rest2 {sid}: {len(matching_rest2)} prefix-matched candidates")
        continue

    rest2_head = matching_rest2[0]

    records.append({
        "subject_id": sid,
        "prefix": prefix,
        "rest1_head": str(rest1_head),
        "rest2_head": str(rest2_head)
    })

scan_manifest = pd.DataFrame(records)

print("Manifest rows:", len(scan_manifest))
print(scan_manifest[["subject_id", "prefix"]].to_string(index=False))

Manifest rows: 22
subject_id prefix
     E3746     DP
     E3799     HL
     E3973     LD
     E4051     SA
     E4209     LS
     E4253     SA
     E4324     JV
     E4350     SF
     E4360     KS
     E4484     CB
     E4673     CL
     E4689     CC
     E4696     DH
     E4697     CM
     E4745     LA
     E5215     RS
     E5376     KE
     E5580     IR
     E5586     PS
     E5653     MM
     E5693     KH
     E5694     AC


### 9.4 Usable-Volume Assessment

In [51]:
import numpy as np

qc_records = []

for _, row in scan_manifest.iterrows():
    sid = row["subject_id"]
    prefix = row["prefix"]

    for session, folder in [("Rest1", REST1_DIR), ("Rest2", REST2_DIR)]:

        # Find censor file matching subject + correct prefix + session
        candidates = [
            p for p in folder.rglob("*censor.1D")
            if sid in str(p)
            and f"{prefix}_{sid}" in p.name
        ]

        if session == "Rest1":
            candidates = [p for p in candidates if "rest2" not in p.name.lower()]
        else:
            candidates = [p for p in candidates if "rest2" in p.name.lower()]

        if len(candidates) != 1:
            print(f"WARNING {sid} {session}: {len(candidates)} censor files")
            continue

        censor_file = candidates[0]

        censor = np.loadtxt(censor_file)
        censor = np.ravel(censor)

        total_volumes = len(censor)
        usable_volumes = int(np.sum(censor))
        censored_volumes = total_volumes - usable_volumes
        percent_censored = 100 * censored_volumes / total_volumes

        qc_records.append({
            "subject_id": sid,
            "session": session,
            "total_volumes": total_volumes,
            "usable_volumes": usable_volumes,
            "censored_volumes": censored_volumes,
            "percent_censored": percent_censored,
            "censor_file": str(censor_file)
        })

qc_df = pd.DataFrame(qc_records)

print("QC rows:", len(qc_df))

print("\nSummary:")
print(
    qc_df.groupby("session")[
        ["total_volumes", "usable_volumes", "censored_volumes", "percent_censored"]
    ].describe()
)

QC rows: 44

Summary:
        total_volumes                                                 \
                count   mean  std    min    25%    50%    75%    max   
session                                                                
Rest1            22.0  260.0  0.0  260.0  260.0  260.0  260.0  260.0   
Rest2            22.0  260.0  0.0  260.0  260.0  260.0  260.0  260.0   

        usable_volumes              ... censored_volumes         \
                 count        mean  ...              75%    max   
session                             ...                           
Rest1             22.0  235.863636  ...            31.25  111.0   
Rest2             22.0  237.863636  ...            12.00  170.0   

        percent_censored                                                \
                   count      mean        std  min       25%       50%   
session                                                                  
Rest1               22.0  9.283217  13.245677  0.0  0.19230

In [53]:
qc_sorted = qc_df.sort_values(
    "percent_censored",
    ascending=False
)

print(
    qc_sorted[
        [
            "subject_id",
            "session",
            "total_volumes",
            "usable_volumes",
            "censored_volumes",
            "percent_censored"
        ]
    ].to_string(
        index=False,
        formatters={
            "percent_censored": lambda x: f"{x:.2f}%"
        }
    )
)

subject_id session  total_volumes  usable_volumes  censored_volumes percent_censored
     E4673   Rest2            260              90               170           65.38%
     E4696   Rest2            260             132               128           49.23%
     E4673   Rest1            260             149               111           42.69%
     E4696   Rest1            260             161                99           38.08%
     E5376   Rest1            260             164                96           36.92%
     E5653   Rest2            260             205                55           21.15%
     E5653   Rest1            260             215                45           17.31%
     E4689   Rest1            260             219                41           15.77%
     E3799   Rest1            260             227                33           12.69%
     E3746   Rest1            260             234                26           10.00%
     E4689   Rest2            260             234                

### 9.5 QC Criterion and Final Analysis Cohort

A scan is considered usable when at least 80% of its fMRI volumes remain
after motion censoring. This corresponds to a maximum of 20% censored
volumes.

For the paired pre-post analysis, a participant must satisfy this criterion
in both Rest 1 and Rest 2. Participants failing the criterion in either
session are excluded from the final imaging analysis cohort. 

In [55]:
MIN_GOOD_TR_FRAC = 0.80

qc_df["good_tr_fraction"] = (
    qc_df["usable_volumes"] / qc_df["total_volumes"]
)

qc_df["passes_qc"] = (
    qc_df["good_tr_fraction"] >= MIN_GOOD_TR_FRAC
)

# Determine participant-level QC status
qc_pivot = qc_df.pivot(
    index="subject_id",
    columns="session",
    values="passes_qc"
)

qc_pivot["final_qc_pass"] = (
    qc_pivot["Rest1"] & qc_pivot["Rest2"]
)

print("Paired participants:", len(qc_pivot))
print("Pass both sessions:", qc_pivot["final_qc_pass"].sum())
print("Fail at least one session:", (~qc_pivot["final_qc_pass"]).sum())

print("\nParticipants failing QC:")
print(qc_pivot.loc[~qc_pivot["final_qc_pass"]])

Paired participants: 22
Pass both sessions: 18
Fail at least one session: 4

Participants failing QC:
session     Rest1  Rest2  final_qc_pass
subject_id                             
E4673       False  False          False
E4696       False  False          False
E5376       False   True          False
E5653        True  False          False


#### QC Finding

Of the 22 participants with paired Rest 1 and Rest 2 imaging data, 18
satisfied the motion-censoring criterion in both sessions.

Four participants failed the criterion in at least one session. E4673 and
E4696 failed in both sessions, E5376 failed in Rest 1, and E5653 failed in
Rest 2.

The resulting QC-passing paired imaging cohort contains 18 participants.

### 9.6 Group Composition After Quality Control

This step links the QC-passing imaging cohort to treatment-group metadata
to determine the active and sham composition of the analysis sample.

In [57]:
final_qc_ids = qc_pivot.index[
    qc_pivot["final_qc_pass"]
].tolist()

final_metadata = paired_metadata[
    paired_metadata["Exam"].isin(final_qc_ids)
].copy()

print("QC-passing participants:", len(final_qc_ids))

print("\nGroup composition:")
print(
    final_metadata[["Exam", "Group"]]
    .drop_duplicates()["Group"]
    .value_counts()
)

print("\nDiagnosis composition:")
print(
    final_metadata[["Exam", "Dx"]]
    .drop_duplicates()["Dx"]
    .value_counts()
)

print("\nExcluded participants and groups:")
print(
    paired_metadata[
        paired_metadata["Exam"].isin(
            qc_pivot.index[~qc_pivot["final_qc_pass"]]
        )
    ][["Exam", "Group", "Dx"]]
    .to_string(index=False)
)

QC-passing participants: 18

Group composition:
Group
active    10
sham       8
Name: count, dtype: int64

Diagnosis composition:
Dx
MDD    18
Name: count, dtype: int64

Excluded participants and groups:
 Exam  Group  Dx
E4673   sham MDD
E4696   sham MDD
E5376 active MDD
E5653   sham MDD


#### Final QC Cohort Finding

After applying the motion-censoring criterion, 18 participants remained
eligible for paired imaging analysis.

The QC-passing cohort contains 10 participants in the active neurofeedback
group and 8 participants in the sham group. All 18 participants have an
MDD diagnosis.

Four participants were excluded because at least one resting-state session
failed the usable-volume criterion: E4673, E4696, E5376, and E5653.

Because exclusions were not evenly distributed across treatment groups,
subsequent active-versus-sham comparisons will account for the resulting
group sizes.